# Compare decoding and retune Qwen, Sailor and Mistral

Attach exactly one completed October16 full notebook output plus the unchanged private78-row grouped-validation dataset. Folder names can vary: saved files are discovered recursively and checked by hashes. Enable Internet, GPU T4 ×2 and the existing GITHUB_TOKEN/HF_TOKEN secrets; add WANDB_API_KEY for live tracking. Save & Run All.

The committed recipe compares eleven decoding strategies, trains one standard-LoRA challenger for each of the three classifiers plus the Qwen corrector, compares native finalists, and exports the selected soft-voting pipeline. Original checkpoints remain candidates. Seed42, fixed data and shared development rows; no folds, hints, quantized training or notebook switches. Four new model fits are intentional;9h experiment budget with bounded stages and memory-release checks.

Monitor the W&B root named run for live metrics, GPU/CPU/RAM resources and heartbeat. Download the complete output, including run/results.zip, run/selected.json and nested model artifacts; preserve the original October16 output too. Local NF4 inference scores are diagnostic only. Native Kaggle fits and any official score improvement remain unverified until execution. Keep all datasets and outputs private.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
REPO_URL = "https://github.com/rquq/GECC_VLSP_26.git"
BRANCH = "kaggle-linux"
SOURCE_ROOT = Path("/kaggle/working")
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Fetch the committed source securely into a fresh directory.
from urllib.parse import urlsplit
import tempfile

with tempfile.TemporaryDirectory() as authentication:
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    token = None
    if urlsplit(REPO_URL).hostname == "github.com":
        from kaggle_secrets import UserSecretsClient
        try:
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            raise RuntimeError("Enable GITHUB_TOKEN in Kaggle Secrets with repository read access.") from None
        if not token:
            raise RuntimeError("GITHUB_TOKEN is empty; add a repository-read token in Kaggle Secrets.")
        askpass = Path(authentication) / "git-askpass.sh"
        askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) printf "%s\\n" x-access-token ;;\n'
                           '*) printf "%s\\n" "$GECC_GITHUB_TOKEN" ;;\nesac\n')
        askpass.chmod(0o700)
        env.update(GIT_ASKPASS=str(askpass), GECC_GITHUB_TOKEN=token)
    PROJECT_DIR = Path(tempfile.mkdtemp(prefix="gecc-source-", dir=SOURCE_ROOT))
    result = subprocess.run(["git", "-c", "credential.helper=", "clone", "--depth", "1",
                             "--single-branch", "--branch", BRANCH, "--", REPO_URL, str(PROJECT_DIR)],
                            env=env, capture_output=True, text=True)
    returncode = result.returncode
    detail = result.stderr.replace(token, "[REDACTED]") if token else result.stderr
    del env, token, result
    if returncode:
        raise RuntimeError("Cannot clone source; check Internet, branch, and GITHUB_TOKEN access. " + detail.strip())
SOURCE_INFO = {"repo_url": REPO_URL, "branch": BRANCH,
               "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR, text=True).strip()}
print(json.dumps(SOURCE_INFO, indent=2))


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("gecc_notebook", PROJECT_DIR / "tools/notebook.py")
launcher = importlib.util.module_from_spec(spec)
spec.loader.exec_module(launcher)
TRAINING_PYTHON = launcher.setup_kaggle(PROJECT_DIR)


In [ ]:
RUN_DIR = launcher.launch(PROJECT_DIR, TRAINING_PYTHON)
